In [1]:
import sys
from pathlib import Path

# resolve imports from the current working dir.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
project_root = next((p for p in candidates if (p / "FEX").exists()), cwd.parent)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from FEX import (
    Controller,
    train_network_controller,
    NumericalDeriv,
    ControllerConfig,
    FEXConfig,
    runtimeconfig,
)

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

Using device: cuda


In [2]:
from pathlib import Path
import logging

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
log_path = log_dir / "train.log"

train_logger = logging.getLogger("train_logger")
train_logger.setLevel(logging.DEBUG)
train_logger.handlers.clear()
train_logger.propagate = False

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(log_path, mode="w")
file_handler.setFormatter(formatter)
file_handler.setLevel(logging.DEBUG)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
stream_handler.setLevel(logging.INFO)

train_logger.addHandler(file_handler)
train_logger.addHandler(stream_handler)

In [ ]:
import pandas as pd
import numpy as np

adj_matrix = pd.read_csv('data/BA_Nnodes100_Adj.csv', header=None)
num_graph_nodes = adj_matrix.shape[0]

x_df = pd.read_csv('data/HR_timeseries_SNR_40.csv', header=None)
num_timesteps, num_cols = x_df.shape
x_np = x_df.to_numpy(dtype=np.float32)
x_data = torch.from_numpy(x_np.reshape(num_timesteps, num_graph_nodes, 3))

dt = 0.01
dx_dt = NumericalDeriv(x_data, dt=dt) # 4th order
x_data = x_data[2:-2]

In [4]:
train_test_split = int(0.8 * len(x_data))
train_x_data = x_data[:train_test_split]
train_dx_dt = dx_dt[:train_test_split]
adj_matrix_tensor = torch.tensor(adj_matrix.values, dtype=torch.float32)
x_data_tensor_ds = TensorDataset(train_x_data, train_dx_dt)
x_data_tensor_ds = x_data_tensor_ds[:len(x_data_tensor_ds)//5]
if runtimeconfig.device == "cuda":
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True, pin_memory=True)
else:
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True)

In [ ]:
from FEX.tree_configs import *
tree_config = get_tree_config("depth_4_config")
controller_config = ControllerConfig(
    input_dim = 20,
    hidden_dim = 64,
    lr = 0.01,
    num_epochs = 50,
    num_cands_per_epoch = 15,
    percentile_threshold = 0.3,
    num_trees = 2
)
fex_config = FEXConfig(
    num_epochs = 20, 
    lr = 0.002,
    bfgs_lr = 0.1,
    max_nodes = 20,
    max_norm = 1.0,
    leaf_dim = x_data.shape[2],

    num_leaves = tree_config.num_leaves
)

best_candidates = train_network_controller(
    tree_config,
    dataloader,
    adj_matrix_tensor,
    controller_config,
    fex_config,
    train_logger=train_logger,
)

2026-03-18 21:22:52,248 | INFO | Epoch 0, Candidate 0, Score: 500887.7188, Reward: 0.0000
2026-03-18 21:23:01,863 | INFO | Epoch 0, Candidate 1, Score: 64589.4375, Reward: 0.0000
2026-03-18 21:23:11,048 | INFO | Epoch 0, Candidate 2, Score: 228659.7344, Reward: 0.0000
2026-03-18 21:23:20,121 | INFO | Epoch 0, Candidate 3, Score: 508207.6562, Reward: 0.0000
2026-03-18 21:23:30,107 | INFO | Epoch 0, Candidate 4, Score: 80955.6016, Reward: 0.0000
2026-03-18 21:23:39,834 | INFO | Epoch 0, Candidate 5, Score: 457859301376.0000, Reward: 0.0000
2026-03-18 21:23:50,578 | INFO | Epoch 0, Candidate 6, Score: 9541.5254, Reward: 0.0001
2026-03-18 21:24:00,744 | INFO | Epoch 0, Candidate 7, Score: 1016616192.0000, Reward: 0.0000
2026-03-18 21:24:04,939 | INFO | Epoch 0, Candidate 8, Score: 57679772.0000, Reward: 0.0000
2026-03-18 21:24:14,547 | INFO | Epoch 0, Candidate 9, Score: 128425.9922, Reward: 0.0000
2026-03-18 21:24:14,567 | INFO | Controller Epoch 0, Loss: 0.0005847170250490308
2026-03-18 

In [6]:
best_candidates.visualize_candidates(directory="logs/candidates_viz")

In [7]:
best_candidates.save_candidates("logs/best_candidates")